# Investigação Social Media — Execução Canônica

Notebook de auditoria do Challenge 004. Os cálculos usam o mesmo `analysis_engine.py` do dashboard e dos relatórios.

**Plataformas:** Instagram, TikTok, YouTube, Bilibili e RedNote.

**Regra de leitura:** rankings são sempre contextualizados com amostra, magnitude, tempo e controles. Performance de patrocínio não é chamada de ROI sem custo/receita/conversão.

In [1]:
from pathlib import Path
import sys, pandas as pd, numpy as np
from IPython.display import display, Markdown

# Funciona dentro da pasta do projeto; em Colab, faça upload/clone do projeto inteiro.
root = Path.cwd()
if not (root / "analysis_engine.py").exists():
    candidates = [root.parent, root.parent.parent, Path('/content')]
    for c in candidates:
        if (c / "analysis_engine.py").exists():
            root = c; break
sys.path.insert(0, str(root))
from analysis_engine import *
DATA = root / "data" / "social_media_dataset.csv"
df = load_data(DATA)
print(f"Base carregada: {len(df):,} posts | {df.data_hora.min():%d/%m/%Y} a {df.data_hora.max():%d/%m/%Y}")

Base carregada: 52,214 posts | 29/05/2023 a 28/05/2025


## 1. Panorama cross-platform

In [2]:
display(platform_overview(df))

,plataforma,posts,inicio,fim,visualizacoes_tipicas,curtidas_tipicas,comentarios_tipicos,compartilhamentos_tipicos,interacoes_tipicas,interacoes_por_100_views,posts_patrocinados_pct
0,Bilibili,10598,2023-05-29 00:18:00,2025-05-28 09:32:00,10100.0,1510.0,200.0,300.0,2010.0,19.904631,42.017362
1,Instagram,10423,2023-05-29 02:53:00,2025-05-28 09:46:00,10100.0,1509.0,200.0,300.0,2009.0,19.888955,41.916915
2,RedNote,10402,2023-05-29 00:15:00,2025-05-28 11:08:00,10101.0,1511.0,200.0,300.0,2010.0,19.900990,43.462796
3,TikTok,10296,2023-05-29 01:07:00,2025-05-28 10:52:00,10101.0,1510.0,200.0,300.0,2010.0,19.901853,43.133256
4,YouTube,10495,2023-05-29 00:58:00,2025-05-28 10:03:00,10099.0,1510.0,200.0,300.0,2010.0,19.900150,43.163411


## 2. Universo de cruzamentos definido antes das conclusões

In [3]:
for bloco in cross_universe():
    display(Markdown(f"### {bloco['familia']}"))
    display(pd.DataFrame({'Cruzamento / pergunta': bloco['cruzamentos']}))

### Objetivos

,Cruzamento / pergunta
0,visualizações
1,curtidas
2,comentários
3,compartilhamentos
4,interações totais
5,curtidas/comentários/compartilhamentos por 100...
6,interações por 100 visualizações


### Conteúdo

,Cruzamento / pergunta
0,formato
1,categoria
2,formato × categoria
3,formato × categoria × tamanho do creator
4,formato × categoria × patrocínio
5,formato × duração relativa
6,categoria × duração relativa


### Creators

...

,Cruzamento / pergunta
0,duração relativa × formato/categoria/tamanho/p...
1,hashtags × formato/categoria
2,quantidade de hashtags × resultados


### Robustez

,Cruzamento / pergunta
0,top/bottom 1%
1,top/bottom 5%
2,média × mediana
3,agregado × controlado
4,efeito de composição / possível paradoxo de Si...
5,primeiro período × segundo período
6,persistência mês a mês
7,teste fora do tempo


### Cross-platform

,Cruzamento / pergunta
0,plataforma × objetivo
1,plataforma × formato
2,plataforma × categoria
3,plataforma × tamanho
4,plataforma × audiência
5,plataforma × patrocínio
6,plataforma × tempo
7,mesma combinação em plataformas diferentes


## 3. Auditoria da base e dos creator IDs

In [4]:
rows=[]
for p in sorted(df.plataforma.unique()):
    d=platform_data(df,p); a=audit_quality(d)
    rows.append({
        'plataforma':p,'posts':a['posts'],
        'variação relativa de views (%)':100*a['distribuicao_metricas']['visualizacoes']['variacao_relativa'],
        'creator IDs com múltiplos nomes (%)':a['creator_ids_multiplos_nomes_pct'],
        'creator IDs com múltiplas contagens (%)':a['creator_ids_multiplas_contagens_pct']
    })
display(pd.DataFrame(rows))

,plataforma,posts,variação relativa de views (%),creator IDs com múltiplos nomes (%),creator IDs com múltiplas contagens (%)
0,Bilibili,10598,0.995485,72.555066,72.555066
1,Instagram,10423,0.988939,71.596452,71.596452
2,RedNote,10402,0.983682,71.953921,71.953921
3,TikTok,10296,0.990807,71.740098,71.740098
4,YouTube,10495,0.992731,72.070485,72.070485


## 4. Objetivos separados: alcance, conversa, compartilhamento, interação e eficiência

In [5]:
metricas=['visualizacoes','comentarios','compartilhamentos','interacoes_totais','taxa_engajamento']
for p in ['Instagram','TikTok','YouTube','Bilibili','RedNote']:
    display(Markdown(f"### {p}"))
    d=platform_data(df,p)
    out=[]
    for m in metricas:
        t=combination_analysis(d,['formato','categoria','porte_criador'],m,80,8,8)
        stable=t[(t.meses_validos>=10)&(t.meses_acima_do_padrao_pct>=60)] if 'meses_validos' in t else pd.DataFrame()
        r=(stable.iloc[0] if not stable.empty else t.iloc[0])
        out.append({
            'objetivo':METRICS[m], 'formato':r.formato,'categoria':r.categoria,'porte_criador':r.porte_criador,
            'posts':int(r.n),'resultado':r.mediana,'diferença_plataforma':r.diferenca_para_plataforma,
            'meses_válidos':r.get('meses_validos',np.nan),'meses_acima_padrão_%':r.get('meses_acima_do_padrao_pct',np.nan),
            'primeiros_12m':r.get('primeiros_12m',np.nan),'segundos_12m':r.get('segundos_12m',np.nan)
        })
    display(pd.DataFrame(out))

### Instagram

,objetivo,formato,categoria,porte_criador,posts,resultado,diferença_plataforma,meses_válidos,meses_acima_padrão_%,primeiros_12m,segundos_12m
0,Visualizações,video,tech,100–250 mil,176,10120.500000,20.500000,13.0,92.307692,10120.000000,10121.500000
1,Comentários,video,tech,100–250 mil,176,204.500000,4.500000,13.0,84.615385,206.500000,201.000000
2,Compartilhamentos,video,lifestyle,100–250 mil,382,304.000000,4.000000,23.0,69.565217,302.000000,304.000000
3,Interações totais,image,lifestyle,250–500 mil,196,2013.000000,4.000000,14.0,71.428571,2009.500000,2015.000000
4,Interações a cada 100 visualizações,video,lifestyle,100–250 mil,382,19.928581,0.039626,23.0,65.217391,19.938068,19.913164


### TikTok

,objetivo,formato,categoria,porte_criador,posts,resultado,diferença_plataforma,meses_válidos,meses_acima_padrão_%,primeiros_12m,segundos_12m
0,Visualizações,video,tech,250–500 mil,295,10114.000000,13.000000,24.0,62.500000,10118.500000,10108.000000
1,Comentários,mixed,lifestyle,500 mil–1 mi,173,202.000000,2.000000,10.0,60.000000,203.000000,201.000000
2,Compartilhamentos,image,beauty,250–500 mil,197,301.000000,1.000000,16.0,62.500000,303.000000,300.000000
3,Interações totais,text,lifestyle,500 mil–1 mi,202,2019.000000,9.000000,15.0,60.000000,2018.000000,2019.000000
4,Interações a cada 100 visualizações,text,lifestyle,500 mil–1 mi,202,19.990194,0.088341,15.0,66.666667,20.032458,19.921066


### YouTube

,objetivo,formato,categoria,porte_criador,posts,resultado,diferença_plataforma,meses_válidos,meses_acima_padrão_%,primeiros_12m,segundos_12m
0,Visualizações,video,lifestyle,100–250 mil,353,10100.000000,1.000000,23.0,60.869565,10095.00000,10107.000000
1,Comentários,image,lifestyle,250–500 mil,227,201.000000,1.000000,16.0,68.750000,198.50000,203.000000
2,Compartilhamentos,mixed,lifestyle,500 mil–1 mi,214,301.500000,1.500000,14.0,78.571429,300.00000,302.000000
3,Interações totais,text,beauty,500 mil–1 mi,197,2016.000000,6.000000,13.0,69.230769,2014.00000,2020.000000
4,Interações a cada 100 visualizações,text,beauty,500 mil–1 mi,197,19.982061,0.081911,13.0,76.923077,19.99021,19.968492


### Bilibili

,objetivo,formato,categoria,porte_criador,posts,resultado,diferença_plataforma,meses_válidos,meses_acima_padrão_%,primeiros_12m,segundos_12m
0,Visualizações,video,beauty,100–250 mil,392,10110.000000,10.00000,24.0,62.500000,10109.0,10110.50000
1,Comentários,video,beauty,500 mil–1 mi,1344,200.000000,0.00000,24.0,62.500000,200.0,201.00000
2,Compartilhamentos,image,lifestyle,500 mil–1 mi,409,303.000000,3.00000,24.0,66.666667,300.5,304.00000
3,Interações totais,mixed,lifestyle,500 mil–1 mi,214,2014.500000,4.50000,17.0,64.705882,2013.0,2015.50000
4,Interações a cada 100 visualizações,image,beauty,500 mil–1 mi,403,19.957941,0.05331,24.0,62.500000,20.0,19.85732


### RedNote

,objetivo,formato,categoria,porte_criador,posts,resultado,diferença_plataforma,meses_válidos,meses_acima_padrão_%,primeiros_12m,segundos_12m
0,Visualizações,video,beauty,100–250 mil,377,10111.000000,10.000000,24.0,70.833333,10103.000000,10115.500000
1,Comentários,video,lifestyle,100–250 mil,349,202.000000,2.000000,24.0,62.500000,202.000000,201.500000
2,Compartilhamentos,video,tech,100–250 mil,177,302.000000,2.000000,11.0,81.818182,295.500000,305.000000
3,Interações totais,image,lifestyle,500 mil–1 mi,449,2016.000000,6.000000,24.0,62.500000,2017.000000,2015.000000
4,Interações a cada 100 visualizações,image,lifestyle,500 mil–1 mi,449,19.964154,0.063164,24.0,75.000000,19.964358,19.950884


## 5. Reconstrução temporal: o que mudou entre janelas equivalentes

In [6]:
for p in ['Instagram','TikTok','YouTube','Bilibili','RedNote']:
    d=platform_data(df,p)
    display(Markdown(f"### {p}"))
    period=d.groupby('periodo_12m').agg(posts=('id','size'),views=('visualizacoes','median'),interacoes=('interacoes_totais','median'),interacoes_por_100_views=('taxa_engajamento','median'),patrocinio_pct=('patrocinado',lambda s:100*s.mean()))
    display(period)
    display(Markdown('**Mudança de mix — categoria**'))
    display(mix_shift(d,'categoria'))
    display(Markdown('**Mudança de mix — formato**'))
    display(mix_shift(d,'formato'))

### Instagram

,posts,views,interacoes,interacoes_por_100_views,patrocinio_pct
periodo_12m,,,,,
Primeiros 12 meses,5239,10100.0,2008.0,19.887787,41.935484
Segundos 12 meses,5184,10102.0,2009.0,19.890846,41.898148


**Mudança de mix — categoria**

periodo_12m,categoria,Primeiros 12 meses,Segundos 12 meses,mudanca_pontos_percentuais
0,beauty,39.301393,41.242284,1.940891
2,tech,20.347395,19.733796,-0.613598
1,lifestyle,40.351212,39.023920,-1.327292


**Mudança de mix — formato**

...

,posts,views,interacoes,interacoes_por_100_views,patrocinio_pct
periodo_12m,,,,,
Primeiros 12 meses,5219,10101.0,2010.0,19.900050,43.590726
Segundos 12 meses,5183,10101.0,2010.0,19.901865,43.333976


**Mudança de mix — categoria**

periodo_12m,categoria,Primeiros 12 meses,Segundos 12 meses,mudanca_pontos_percentuais
0,beauty,40.237593,40.922246,0.684652
1,lifestyle,39.662771,39.648852,-0.013919
2,tech,20.099636,19.428902,-0.670734


**Mudança de mix — formato**

periodo_12m,formato,Primeiros 12 meses,Segundos 12 meses,mudanca_pontos_percentuais
3,video,59.609121,60.679143,1.070023
0,image,20.003832,20.239244,0.235412
2,text,9.542058,8.913757,-0.628301
1,mixed,10.844989,10.167856,-0.677133


## 6. Patrocínio: agregado e controlado

In [7]:
for p in ['Instagram','TikTok','YouTube','Bilibili','RedNote']:
    d=platform_data(df,p)
    display(Markdown(f"### {p} — comparação geral"))
    display(sponsorship_overall(d,['visualizacoes','comentarios','compartilhamentos','interacoes_totais','taxa_engajamento'],n_boot=300))
    display(Markdown('**Exemplos controlados por formato × categoria × tamanho do creator (taxa de engajamento)**'))
    ctrl=sponsorship_controlled(d,'taxa_engajamento',['formato','categoria','porte_criador'],25,with_ci=False)
    display(pd.concat([ctrl.head(5),ctrl.tail(5)]).drop_duplicates())

### Instagram — comparação geral

,metrica,n_patrocinado,n_organico,resultado_patrocinado,resultado_organico,diferenca,intervalo_95_min,intervalo_95_max,efeito_ordinal,evidencia
0,visualizacoes,4369,6054,10100.00000,10101.000000,-1.000000,-4.000000,4.525000,-0.008418,diferença não separada de zero
1,comentarios,4369,6054,200.00000,200.000000,0.000000,0.000000,2.000000,0.017028,diferença não separada de zero
2,compartilhamentos,4369,6054,299.00000,300.000000,-1.000000,-2.000000,0.000000,-0.022698,diferença não separada de zero
3,interacoes_totais,4369,6054,2009.00000,2008.000000,1.000000,-1.000000,4.000000,0.012552,diferença não separada de zero
4,taxa_engajamento,4369,6054,19.89917,19.881174,0.017996,-0.006111,0.039281,0.013314,diferença não separada de zero


**Exemplos controlados por formato × categoria × tamanho do creator (taxa de engajamento)**

,formato,categoria,porte_criador,n_patrocinado,n_organico,resultado_patrocinado,resultado_organico,diferenca,intervalo_95_min,intervalo_95_max,efeito_ordinal,evidencia
0,video,beauty,50–100 mil,47,78,20.010020,19.798619,0.211401,NaN,NaN,0.158211,comparação descritiva controlada
1,image,tech,500 mil–1 mi,96,114,19.947196,19.825882,0.121313,NaN,NaN,0.142452,comparação descritiva controlada
2,image,lifestyle,100–250 mil,57,81,19.918457,19.809301,0.109155,NaN,NaN,0.070825,comparação descritiva controlada
3,text,beauty,250–500 mil,47,55,20.142872,20.034041,0.108831,NaN,NaN,-0.013540,comparação descritiva controlada
4,mixed,beauty,250–500 mil,51,67,20.037472,19.932163,0.105309,NaN,NaN,0.113257,comparação descritiva controlada
29,video,tech,50–100 mil,29,40,19.839840,19.884349,-0.044510,NaN,NaN,0.024138,comparação descritiva controlada
30,image,lifestyle,250–500 mil,94,102,19.858763,19.938534,-0.079771,NaN,NaN,-0.065081,comparação descritiva controlada
31,text,beauty,500 mil–1 mi,98,122,19.785595,19.913152,-0.127557,NaN,NaN,-0.039394,comparação descritiva controlada
32,video,beauty,100–250 mil,156,270,19.796784,19.950298,-0.153515,NaN,NaN,-0.166477,comparação descritiva controlada
33,video,lifestyle,10–50 mil,29,71,19.710881,19.947069,-0.236188,NaN,NaN,-0.097620,comparação descritiva controlada


### TikTok — comparação geral

...

,formato,categoria,porte_criador,n_patrocinado,n_organico,resultado_patrocinado,resultado_organico,diferenca,intervalo_95_min,intervalo_95_max,efeito_ordinal,evidencia
0,image,lifestyle,100–250 mil,46,79,19.901656,19.760182,0.141473,NaN,NaN,0.151348,comparação descritiva controlada
1,video,tech,100–250 mil,82,131,19.991159,19.873443,0.117716,NaN,NaN,0.141128,comparação descritiva controlada
2,text,tech,500 mil–1 mi,47,55,19.934995,19.840802,0.094193,NaN,NaN,0.029014,comparação descritiva controlada
3,mixed,tech,500 mil–1 mi,44,62,19.925231,19.843931,0.081300,NaN,NaN,0.002199,comparação descritiva controlada
4,image,beauty,500 mil–1 mi,168,235,19.998031,19.917094,0.080937,NaN,NaN,0.075887,comparação descritiva controlada
33,mixed,beauty,250–500 mil,43,64,19.899170,19.997935,-0.098765,NaN,NaN,-0.047238,comparação descritiva controlada
34,text,lifestyle,500 mil–1 mi,86,115,19.883526,20.003965,-0.120440,NaN,NaN,-0.168453,comparação descritiva controlada
35,text,beauty,100–250 mil,38,40,19.829441,19.981157,-0.151716,NaN,NaN,-0.159211,comparação descritiva controlada
36,video,beauty,50–100 mil,57,80,19.698933,19.900548,-0.201614,NaN,NaN,-0.160088,comparação descritiva controlada
37,video,tech,50–100 mil,32,43,19.798713,20.030151,-0.231437,NaN,NaN,-0.225291,comparação descritiva controlada


### RedNote — comparação geral

,metrica,n_patrocinado,n_organico,resultado_patrocinado,resultado_organico,diferenca,intervalo_95_min,intervalo_95_max,efeito_ordinal,evidencia
0,visualizacoes,4521,5881,10100.000000,10102.000000,-2.000000,-7.000000,3.000000,-0.003583,diferença não separada de zero
1,comentarios,4521,5881,200.000000,200.000000,0.000000,-1.000000,1.000000,0.022070,diferença não separada de zero
2,compartilhamentos,4521,5881,300.000000,299.000000,1.000000,-1.000000,1.000000,0.012862,diferença não separada de zero
3,interacoes_totais,4521,5881,2011.000000,2010.000000,1.000000,-1.000000,3.525000,0.010490,diferença não separada de zero
4,taxa_engajamento,4521,5881,19.910935,19.894716,0.016219,-0.003531,0.035543,0.014139,diferença não separada de zero


**Exemplos controlados por formato × categoria × tamanho do creator (taxa de engajamento)**

,formato,categoria,porte_criador,n_patrocinado,n_organico,resultado_patrocinado,resultado_organico,diferenca,intervalo_95_min,intervalo_95_max,efeito_ordinal,evidencia
0,image,lifestyle,100–250 mil,42,58,20.028818,19.776343,0.252475,NaN,NaN,0.316092,comparação descritiva controlada
1,text,lifestyle,250–500 mil,48,56,20.052647,19.826310,0.226337,NaN,NaN,0.186756,comparação descritiva controlada
2,video,beauty,10–50 mil,49,56,19.938407,19.761321,0.177085,NaN,NaN,0.116618,comparação descritiva controlada
3,video,tech,100–250 mil,75,102,19.985996,19.829587,0.156409,NaN,NaN,0.153987,comparação descritiva controlada
4,mixed,lifestyle,100–250 mil,31,40,20.040221,19.887034,0.153187,NaN,NaN,0.124194,comparação descritiva controlada
31,mixed,tech,500 mil–1 mi,47,67,19.941003,20.013878,-0.072875,NaN,NaN,-0.014925,comparação descritiva controlada
32,video,beauty,50–100 mil,53,72,19.837859,19.914984,-0.077125,NaN,NaN,-0.136792,comparação descritiva controlada
33,image,lifestyle,250–500 mil,87,123,19.863681,19.962484,-0.098803,NaN,NaN,-0.079525,comparação descritiva controlada
34,video,tech,10–50 mil,30,27,19.693173,19.836502,-0.143329,NaN,NaN,-0.059259,comparação descritiva controlada
35,image,tech,100–250 mil,38,38,19.818318,20.084415,-0.266097,NaN,NaN,-0.253463,comparação descritiva controlada


## 7. Audiência

In [8]:
for p in ['Instagram','TikTok','YouTube','Bilibili','RedNote']:
    d=platform_data(df,p)
    display(Markdown(f"### {p}"))
    rows=[]
    for dim in ['faixa_etaria_audiencia','genero_audiencia','localizacao_audiencia','idioma']:
        t=group_summary(d,dim,'taxa_engajamento',30)
        if not t.empty:
            rows.append({'dimensão':DIMENSION_LABELS[dim],'melhor':t.iloc[0][dim],'resultado_melhor':t.iloc[0].mediana,'pior':t.iloc[-1][dim],'resultado_pior':t.iloc[-1].mediana,'distância':t.iloc[0].mediana-t.iloc[-1].mediana})
    display(pd.DataFrame(rows))

### Instagram

,dimensão,melhor,resultado_melhor,pior,resultado_pior,distância
0,Idade predominante da audiência,26-35,19.899788,36-50,19.866247,0.033541
1,Gênero predominante da audiência,non-binary,19.896741,male,19.883450,0.013291
2,Localização predominante da audiência,Brazil,19.907271,Russia,19.870735,0.036536
3,Idioma,Spanish,19.905260,Chinese,19.881722,0.023538


### TikTok

,dimensão,melhor,resultado_melhor,pior,resultado_pior,distância
0,Idade predominante da audiência,26-35,19.918920,50+,19.883519,0.035401
1,Gênero predominante da audiência,unknown,19.915550,female,19.886023,0.029527
2,Localização predominante da audiência,Russia,19.938738,Germany,19.884151,0.054588
3,Idioma,Spanish,19.933821,English,19.890873,0.042948


### YouTube

,dimensão,melhor,resultado_melhor,pior,resultado_pior,distância
0,Idade predominante da audiência,50+,19.909728,19-25,19.898009,0.011719
1,Gênero predominante da audiência,male,19.912167,female,19.890511,0.021656
2,Localização predominante da audiência,Russia,19.916766,Germany,19.885607,0.031159
3,Idioma,Hindi,19.918900,Spanish,19.891519,0.027381


### Bilibili

,dimensão,melhor,resultado_melhor,pior,resultado_pior,distância
0,Idade predominante da audiência,50+,19.929540,26-35,19.896487,0.033053
1,Gênero predominante da audiência,non-binary,19.923224,male,19.893847,0.029377
2,Localização predominante da audiência,UK,19.948418,Japan,19.882841,0.065576
3,Idioma,Spanish,19.941154,English,19.897406,0.043748


### RedNote

,dimensão,melhor,resultado_melhor,pior,resultado_pior,distância
0,Idade predominante da audiência,36-50,19.916559,50+,19.859497,0.057062
1,Gênero predominante da audiência,non-binary,19.914263,female,19.893268,0.020995
2,Localização predominante da audiência,Japan,19.920988,India,19.865680,0.055307
3,Idioma,Chinese,19.921992,English,19.888687,0.033305


## 8. Outliers — top/bottom 1% e 5%

In [9]:
for p in ['Instagram','TikTok','YouTube','Bilibili','RedNote']:
    d=platform_data(df,p)
    display(Markdown(f"### {p}"))
    for pct in [.01,.05]:
        x=outlier_profile(d,'taxa_engajamento',pct)
        display(Markdown(f"**Top/bottom {int(pct*100)}% — segmentos mais super-representados no topo**"))
        display(x.sort_values('indice_top',ascending=False).head(12))

### Instagram

**Top/bottom 1% — segmentos mais super-representados no topo**

,metrica,faixa_extrema,dimensao,segmento,participacao_base_pct,participacao_top_pct,participacao_bottom_pct,indice_top,indice_bottom,top_n,bottom_n
12,taxa_engajamento,0.01,porte_criador,<10 mil,0.853881,1.904762,0.952381,2.230712,1.115356,2,1
8,taxa_engajamento,0.01,porte_criador,10–50 mil,3.770508,6.666667,0.952381,1.768109,0.252587,7,1
22,taxa_engajamento,0.01,genero_audiencia,unknown,10.390483,17.142857,18.095238,1.649861,1.741520,18,19
1,taxa_engajamento,0.01,formato,mixed,9.987528,14.285714,9.523810,1.430355,0.953570,15,10
24,taxa_engajamento,0.01,localizacao_audiencia,Brazil,12.357287,16.190476,9.523810,1.310197,0.770704,17,10
35,taxa_engajamento,0.01,idioma,Japanese,9.786050,12.380952,11.428571,1.265163,1.167843,13,12
38,taxa_engajamento,0.01,duracao_relativa,Longa,19.994243,24.761905,20.000000,1.238452,1.000288,26,21
50,taxa_engajamento,0.01,dia_semana,Sexta,14.170584,17.142857,16.190476,1.209749,1.142541,18,17
28,taxa_engajamento,0.01,localizacao_audiencia,Japan,11.896767,14.285714,9.523810,1.200806,0.800538,15,10
43,taxa_engajamento,0.01,faixa_horaria,06–11,24.724168,29.523810,25.714286,1.194128,1.040047,31,27


**Top/bottom 5% — segmentos mais super-representados no topo**

,metrica,faixa_extrema,dimensao,segmento,participacao_base_pct,participacao_top_pct,participacao_bottom_pct,indice_top,indice_bottom,top_n,bottom_n
35,taxa_engajamento,0.05,idioma,Japanese,9.786050,12.068966,8.795411,1.233283,0.898770,63,46
19,taxa_engajamento,0.05,faixa_etaria_audiencia,50+,5.094503,6.130268,4.015296,1.203310,0.788163,32,21
50,taxa_engajamento,0.05,dia_semana,Sexta,14.170584,16.858238,13.766730,1.189664,0.971501,88,72
23,taxa_engajamento,0.05,genero_audiencia,unknown,10.390483,12.260536,12.810707,1.179978,1.232927,64,67
1,taxa_engajamento,0.05,formato,mixed,9.987528,11.685824,8.604207,1.170042,0.861495,61,45
7,taxa_engajamento,0.05,porte_criador,100–250 mil,15.408232,17.624521,14.531549,1.143838,0.943103,92,76
12,taxa_engajamento,0.05,porte_criador,<10 mil,0.853881,0.957854,0.573614,1.121766,0.671773,5,3
24,taxa_engajamento,0.05,localizacao_audiencia,Brazil,12.357287,13.793103,11.472275,1.116192,0.928381,72,60
28,taxa_engajamento,0.05,localizacao_audiencia,Japan,11.896767,13.026820,12.428298,1.094988,1.044679,68,65
14,taxa_engajamento,0.05,patrocinado,True,41.916915,45.593870,43.785851,1.087720,1.044587,238,229


...

### RedNote

**Top/bottom 1% — segmentos mais super-representados no topo**

,metrica,faixa_extrema,dimensao,segmento,participacao_base_pct,participacao_top_pct,participacao_bottom_pct,indice_top,indice_bottom,top_n,bottom_n
8,taxa_engajamento,0.01,porte_criador,10–50 mil,4.085753,7.619048,2.857143,1.864784,0.699294,8,3
52,taxa_engajamento,0.01,dia_semana,Terça,14.679869,20.952381,15.238095,1.427287,1.038027,22,16
34,taxa_engajamento,0.01,idioma,Hindi,10.276870,13.333333,9.523810,1.297412,0.926723,14,10
15,taxa_engajamento,0.01,faixa_etaria_audiencia,13-18,14.949048,19.047619,14.285714,1.274169,0.955627,20,15
22,taxa_engajamento,0.01,genero_audiencia,non-binary,9.853874,12.380952,9.523810,1.256455,0.966504,13,10
30,taxa_engajamento,0.01,localizacao_audiencia,UK,12.199577,15.238095,5.714286,1.249068,0.468400,16,6
35,taxa_engajamento,0.01,idioma,Japanese,9.959623,12.380952,7.619048,1.243115,0.764994,13,8
39,taxa_engajamento,0.01,duracao_relativa,Muito curta,20.005768,23.809524,21.904762,1.190133,1.094922,25,23
0,taxa_engajamento,0.01,formato,image,20.121131,23.809524,24.761905,1.183309,1.230642,25,26
43,taxa_engajamento,0.01,faixa_horaria,06–11,24.793309,28.571429,23.809524,1.152385,0.960321,30,25


**Top/bottom 5% — segmentos mais super-representados no topo**

,metrica,faixa_extrema,dimensao,segmento,participacao_base_pct,participacao_top_pct,participacao_bottom_pct,indice_top,indice_bottom,top_n,bottom_n
12,taxa_engajamento,0.05,porte_criador,<10 mil,0.874832,1.343570,0.575816,1.535804,0.658202,7,3
24,taxa_engajamento,0.05,localizacao_audiencia,Brazil,12.449529,15.738964,13.819578,1.264222,1.110048,82,72
22,taxa_engajamento,0.05,genero_audiencia,non-binary,9.853874,11.900192,9.596929,1.207666,0.973924,62,50
47,taxa_engajamento,0.05,dia_semana,Quarta,14.054989,16.698656,13.435701,1.188095,0.955938,87,70
11,taxa_engajamento,0.05,porte_criador,10–50 mil,4.085753,4.798464,4.030710,1.174438,0.986528,25,21
2,taxa_engajamento,0.05,formato,text,9.228994,10.556622,7.869482,1.143854,0.852691,55,41
9,taxa_engajamento,0.05,porte_criador,100–250 mil,14.612575,16.506718,14.587332,1.129624,0.998273,86,76
25,taxa_engajamento,0.05,localizacao_audiencia,China,12.747549,14.395393,11.708253,1.129268,0.918471,75,61
50,taxa_engajamento,0.05,dia_semana,Sexta,14.112671,15.930902,13.627639,1.128837,0.965631,83,71
42,taxa_engajamento,0.05,faixa_horaria,00–05,25.447029,28.406910,23.608445,1.116315,0.927749,148,123


## 9. Hashtags e duração relativa

In [10]:
for p in ['Instagram','TikTok','YouTube','Bilibili','RedNote']:
    d=platform_data(df,p)
    display(Markdown(f"### {p}"))
    display(Markdown('**Duração relativa**'))
    display(group_summary(d,'duracao_relativa','taxa_engajamento',30))
    display(Markdown('**Hashtags com pelo menos 30 ocorrências**'))
    display(hashtag_analysis(d,'taxa_engajamento',30).head(20))

### Instagram

**Duração relativa**

,duracao_relativa,n,media,mediana,desvio,p05,p25,p75,p95,amplitude_central,diferenca_para_plataforma,diferenca_relativa_pct
0,Muito curta,2085,19.912872,19.907179,0.486009,19.121571,19.572287,20.244363,20.725552,0.672075,0.018224,0.091628
1,Curta,2084,19.895664,19.886683,0.495781,19.087104,19.557794,20.223193,20.721319,0.665400,-0.002272,-0.011424
2,Muito longa,2085,19.899311,19.885126,0.496570,19.069799,19.560483,20.236220,20.727432,0.675737,-0.003829,-0.019253
3,Longa,2084,19.898374,19.881984,0.495167,19.117598,19.554566,20.230890,20.715367,0.676324,-0.006971,-0.035051
4,Média,2085,19.890031,19.877191,0.488866,19.108096,19.562636,20.216356,20.712561,0.653720,-0.011764,-0.059147


**Hashtags com pelo menos 30 ocorrências**

,hashtag,n,media,mediana,p25,p75,diferenca_para_plataforma
0,before,30,20.033926,20.145878,19.840421,20.346318,0.256923
1,lot,34,20.070134,20.138199,19.624415,20.456184,0.249244
2,relate,33,20.072833,20.134494,19.601823,20.460637,0.245539
3,including,32,20.054030,20.130467,19.711185,20.348575,0.241512
4,same,30,20.073447,20.122369,19.900406,20.341768,0.233414
5,whose,32,20.040071,20.113973,19.596895,20.374527,0.225018
6,federal,31,20.027120,20.108002,19.716407,20.310074,0.219047
7,understand,37,20.061584,20.101050,19.818397,20.349762,0.212095
8,left,32,20.021889,20.095734,19.820216,20.298507,0.206779
9,foot,32,19.996423,20.085075,19.681036,20.268653,0.196120


...

### RedNote

**Duração relativa**

,duracao_relativa,n,media,mediana,desvio,p05,p25,p75,p95,amplitude_central,diferenca_para_plataforma,diferenca_relativa_pct
0,Média,2080,19.918538,19.913505,0.491874,19.093231,19.593498,20.227184,20.757220,0.633685,0.012515,0.062886
1,Muito longa,2081,19.919818,19.913206,0.488551,19.116775,19.588235,20.256719,20.713578,0.668484,0.012216,0.061386
2,Longa,2080,19.898279,19.898970,0.479721,19.125283,19.571835,20.216469,20.706240,0.644634,-0.002020,-0.010151
3,Curta,2080,19.907525,19.893651,0.487377,19.128059,19.570283,20.239208,20.736835,0.668925,-0.007339,-0.036877
4,Muito curta,2081,19.904646,19.881335,0.486718,19.127583,19.583374,20.235387,20.717814,0.652013,-0.019655,-0.098762


**Hashtags com pelo menos 30 ocorrências**

,hashtag,n,media,mediana,p25,p75,diferenca_para_plataforma
0,upon,40,20.123220,20.184247,19.805234,20.439301,0.283257
1,listen,34,20.103732,20.176783,19.797993,20.420739,0.275793
2,stay,33,20.118115,20.154044,19.617367,20.552147,0.253054
3,morning,36,20.072170,20.146859,19.749530,20.319758,0.245869
4,from,35,20.071744,20.128332,19.709631,20.367926,0.227342
5,security,31,20.055433,20.110664,19.799267,20.300017,0.209674
6,fly,31,20.013091,20.109344,19.779303,20.338836,0.208354
7,management,35,20.020202,20.105190,19.763314,20.326988,0.204200
8,ready,34,20.159397,20.093209,19.730976,20.540023,0.192219
9,nation,30,20.055823,20.092453,19.723778,20.387770,0.191463


## 10. Frequência semanal

In [11]:
rows=[]
for p in ['Instagram','TikTok','YouTube','Bilibili','RedNote']:
    d=platform_data(df,p)
    for m in ['visualizacoes','taxa_engajamento']:
        r=frequency_performance(d,m)
        rows.append({'plataforma':p,'métrica':METRICS[m],'relação volume semanal × resultado':r['correlacao_volume_resultado']})
display(pd.DataFrame(rows))
print('Leitura: valores próximos de zero indicam que o histórico não sustenta uma frequência semanal ótima por si só.')

,plataforma,métrica,relação volume semanal × resultado
0,Instagram,Visualizações,-0.122854
1,Instagram,Interações a cada 100 visualizações,0.030294
2,TikTok,Visualizações,-0.046796
3,TikTok,Interações a cada 100 visualizações,-0.149266
4,YouTube,Visualizações,-0.000493
5,YouTube,Interações a cada 100 visualizações,0.059879
6,Bilibili,Visualizações,0.026587
7,Bilibili,Interações a cada 100 visualizações,0.128802
8,RedNote,Visualizações,-0.108726
9,RedNote,Interações a cada 100 visualizações,0.092513


Leitura: valores próximos de zero indicam que o histórico não sustenta uma frequência semanal ótima por si só.


## 11. Stress test fora do tempo

In [12]:
rows=[]
for p in ['Instagram','TikTok','YouTube','Bilibili','RedNote']:
    d=platform_data(df,p)
    for m in ['taxa_engajamento','visualizacoes']:
        r=temporal_predictability(d,m)
        rows.append({'plataforma':p,'métrica':METRICS[m],**r})
display(pd.DataFrame(rows))
print('Leitura: AUC perto de 0,50 significa que as características disponíveis no primeiro período não reconhecem os posts de maior performance no segundo melhor que aproximadamente o acaso.')

,plataforma,métrica,metrica,posts_treino,posts_teste,capacidade_explicar_resultado_futuro,capacidade_identificar_top_20_futuro,limiar_top_20_treino
0,Instagram,Interações a cada 100 visualizações,taxa_engajamento,5239,5184,-0.010729,0.484100,20.311394
1,Instagram,Visualizações,visualizacoes,5239,5184,-0.011671,0.511669,10186.000000
2,TikTok,Interações a cada 100 visualizações,taxa_engajamento,5035,5261,-0.011202,0.508186,20.318957
3,TikTok,Visualizações,visualizacoes,5035,5261,-0.008768,0.507654,10187.000000
4,YouTube,Interações a cada 100 visualizações,taxa_engajamento,5242,5253,-0.006327,0.491867,20.306041
5,YouTube,Visualizações,visualizacoes,5242,5253,-0.006409,0.509571,10184.000000
6,Bilibili,Interações a cada 100 visualizações,taxa_engajamento,5357,5241,-0.004347,0.488988,20.316910
7,Bilibili,Visualizações,visualizacoes,5357,5241,-0.005820,0.520452,10183.000000
8,RedNote,Interações a cada 100 visualizações,taxa_engajamento,5219,5183,-0.008350,0.501427,20.319279
9,RedNote,Visualizações,visualizacoes,5219,5183,-0.015244,0.480332,10184.000000


Leitura: AUC perto de 0,50 significa que as características disponíveis no primeiro período não reconhecem os posts de maior performance no segundo melhor que aproximadamente o acaso.


## 12. Síntese metodológica

- Os cálculos deste notebook são os mesmos do dashboard e dos relatórios.
- As tabelas completas continuam exploráveis no Streamlit até o registro bruto.
- O relatório traduz os resultados para linguagem de negócio; este notebook preserva a camada de auditoria.
- A falta de gasto/receita/conversão impede ROI financeiro.
- Creator ID inconsistente não é usado como identidade longitudinal.
- Horário não vira prescrição sem timezone.
- Content length é interpretado relativamente, sem assumir unidade.